# Chapter 3 — Reasoning Traces Are Not Evidence

*Free-form thoughts are working artifacts. Structured claims are evidence.*

## Objective

Establish early that model-generated reasoning text is a working artifact and not proof. Everything downstream relies on structured claims, tool results and verifications.

| Object | Trust level |
| --- | --- |
| Internal reasoning text | low |
| Tool result | medium |
| Verified tool result | higher |
| Cited source | higher |
| Schema-valid output | higher |
| Independently checked claim | highest |

In [ ]:
from glassloop.reasoning import Entry, EntryType, Scratchpad, TrustLevel
from pydantic import ValidationError

## The unstructured approach

An LM might produce something like the string below. There is no way for the rest of the system to know which sentence is a claim, which is an assumption and which is an observation.

In [ ]:
free_form = (
    'The overdraft fee is $35. It applies once per occurrence. '
    'I think the customer was charged twice in the same day. '
    'Bank policy probably allows a one-time goodwill reversal.'
)
print(free_form)

Useful for the model, useless for a verifier.

## The structured approach

Same content as typed entries. Each carries a kind, an evidence pointer and a trust level. Observations require a source; entries with trust above LOW require evidence. These invariants are enforced in code.

In [ ]:
pad = Scratchpad()
pad.add_claim('The overdraft fee is $35', evidence='overdraft.txt', trust=TrustLevel.HIGH)
pad.add_claim('It applies once per occurrence', evidence='overdraft.txt', trust=TrustLevel.HIGH)
pad.add_assumption('The customer was charged twice in the same day')
pad.add_observation('Customer service log shows two charges on 2026-05-01', source='ticket-1042')
pad.add_question('Was a goodwill reversal already issued this year?')

print(pad.render_table(as_string=True))

## Verification

Now that entries are typed we can ask cheap, real questions about them.

In [ ]:
print('claims:        ', len(pad.by_type(EntryType.CLAIM)))
print('observations:  ', len(pad.by_type(EntryType.OBSERVATION)))
print('open questions:', len(pad.by_type(EntryType.QUESTION)))
print('unsupported claims:', [e.text for e in pad.unsupported_claims()])

pad.assert_all_claims_have_evidence()
print('OK — every claim is supported')

## Invariants are enforced at construction time

An observation without a source — or a high-trust entry without evidence — fails immediately, not silently downstream.

In [ ]:
try:
    Entry(kind=EntryType.OBSERVATION, text='x')
except ValidationError as e:
    print('observation rejected:', e.errors()[0]['msg'])

try:
    Entry(kind=EntryType.CLAIM, text='x', trust=TrustLevel.HIGH)
except ValidationError as e:
    print('high-trust no-evidence rejected:', e.errors()[0]['msg'])

## Anti-patterns flagged here

- Logging the model's chain-of-thought and calling it audit.
- Asserting on the *presence* of a reasoning string.
- Confusing assumptions with observations.

In [ ]:
# Self-check
assert pad.unsupported_claims() == []
assert len(pad.by_type(EntryType.CLAIM)) == 2
assert len(pad.by_type(EntryType.OBSERVATION)) == 1
print('OK')